![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 43 -- Competition: Damaged Archive

A photo archive kept **32 grey photographs**. Every photograph was copied many times, and every copy was changed at random before it was stored. Then all the copies were shuffled together. Your goal is to sort the copies back into **32 groups**, one group for each original photograph.

**How a copy is stored.** To save space, a copy is not stored as a picture. It is stored as two small arrays of numbers: row `i` of `X1` (shape 128 x 4) and row `i` of `X2` (shape 4 x 128) belong to the same copy `i`.

**Score.** Your groups are compared with the true groups by the **Adjusted Rand Index** (`sklearn.metrics.adjusted_rand_score`). Only which copies you put together counts, not the numbers you give the groups. 1 is a perfect grouping; a random grouping scores about 0. Kaggle shows the score on 40% of the copies during the competition and ranks you at the end on the other 60%.

**The one rule of the task.** If you use a neural network, it must be `EmbNet` below, exactly as it is: do not change the class, do not train it, do not fine-tune its weights. Methods without a network (NumPy, scikit-learn) are allowed. No other pretrained model and no other data may be used.

**Your file.** `submission.csv` with the columns `id` and `target`: one row for every copy, `target` the whole number of its group, in the format of `sample_submission.csv`.

**Dataset:** Made from real photographs (credit and licence in the competition's README). The setup cell leaves three files in `DATA_DIR`: `X1.npy` and `X2.npy` (NumPy arrays of `float32`; row `i` of both is copy `i`) and `sample_submission.csv` (the format of the file you hand in).

The clock started at the opening time, so start at once.

---
# Setup

Run the four cells below in order. The first one installs the checkpoint library where it is missing. The second one opens your session with the scorer: replace `PASTE-YOUR-ID` by your own student id before you run it. From that cell on, this notebook records cell runs, cell source and errors for assessment. The third one imports the libraries and the fourth one gets the data.

Your id is yours alone. Do not share a copy of this notebook that holds it.

In [ ]:
# Installs the checkpoint library where it is missing. On Colab that takes a few seconds.
import importlib.util

if importlib.util.find_spec("olympiad") is None:
    !pip -q install olympiad-lab timm
    importlib.invalidate_caches()


In [ ]:
import os
from olympiad import ck

STUDENT_ID = os.environ.get("OLYMPIAD_STUDENT_ID", "PASTE-YOUR-ID")
ck.start(lab="uns-comp", student_id=STUDENT_ID)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
import torch
from torch import nn
import timm

np.random.seed(0)
torch.manual_seed(0)
print("torch", torch.__version__, "| timm", timm.__version__)


In [ ]:
import os

DATA_DIR = os.environ.get("OLYMPIAD_DATA_DIR")      # set when the instructor runs the notebook again
if not DATA_DIR:
    import kagglehub
    DATA_DIR = kagglehub.dataset_download("sattamjaltwaim/damaged-archive")
print("Data folder:", DATA_DIR)
print(sorted(os.listdir(DATA_DIR))[:20])


---
# The network

The only neural network this task allows.

In [ ]:
class EmbNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = timm.create_model("tiny_vit_5m_224.dist_in22k_ft_in1k", pretrained=True, num_classes=0)

    def forward(self, image):
        x = self.model(image)
        return x


---
# Baseline

The starter solution: KMeans on the raw arrays. Its score is the baseline: your points count from there upward.

In [ ]:
def generate_submit(pred_cluster):
    sub = pd.DataFrame()
    sub["id"] = np.arange(len(pred_cluster))
    sub["target"] = pred_cluster
    sub.to_csv("submission.csv", index=False)
    print(sub.head(10))


In [ ]:
X_1 = np.load(os.path.join(DATA_DIR, "X1.npy"))
X_2 = np.load(os.path.join(DATA_DIR, "X2.npy"))
print(X_1.shape, X_2.shape, X_1.dtype)


In [ ]:
km = KMeans(32, n_init=10, random_state=0)
X = np.concatenate((X_1.reshape((X_1.shape[0], X_1.shape[1] * X_1.shape[2])),
                    X_2.reshape((X_2.shape[0], X_2.shape[1] * X_2.shape[2]))), 1)
pred_cluster = km.fit_predict(X)

generate_submit(pred_cluster)


In [ ]:
ck.submit("submission.csv")


---
# Hand in

**Every file goes to Kaggle.** `ck.submit` checks the format of `submission.csv` and records it; the score comes from Kaggle. On Colab, download the file first: open the Files panel (the folder icon on the left), find `submission.csv`, click the three dots beside it, then Download. On the competition's Kaggle page click **Submit Prediction**, choose the file and click **Submit**. Do the same, and run `ck.submit("submission.csv")` again, each time you write a better file.

**Your own cells go below**, as many as you need, above the Finish cell.

**Your final notebook** must run from top to bottom in under 20 minutes, and the file it writes last is the one you choose as your final submission on Kaggle.

In [ ]:
# Your code here


---
# Finish

Run the last cell when you are done. It sends what is still waiting and hands in this notebook.

In [ ]:
ck.finish()

---

Made By **Sattam Altwaim**